In [1]:
import time
import pandas as pd
import os
import joblib

In [2]:
# feature columns
X_cols = [
    "hour","dayofweek","is_weekend","month",
    # "lag_24","rolling_24",
    # "electricity_current",
    "airTemperature", "dewTemperature", "windSpeed",
    # "temp_lag_1h","dewTemperature_lag_1h", "windSpeed_lag_1h",
    # "sqft", 
    "sqm", "primaryspaceusage", "site_id", "building_id",
    "Chilledwater", "Hotwater"
]

In [3]:
model_dir = "models_1578_csv"
os.makedirs(model_dir, exist_ok=True)

In [4]:
def load_data(path):
    data_df = pd.read_csv(path)
    return data_df

# TRAINING


In [5]:

data_encoded = "data_1578_csv/train_encode.csv"
data_frame = load_data(data_encoded)

In [6]:
# data_frame[["primaryspaceusage", "site_id", "buiding_id"]].head()
data_frame

,timestamp,Electricity,airTemperature,dewTemperature,windSpeed,Chilledwater,Hotwater,hour,dayofweek,is_weekend,...,target_t+15,target_t+16,target_t+17,target_t+18,target_t+19,target_t+20,target_t+21,target_t+22,target_t+23,target_t+24
0,2016-01-01 00:00:00,109.00,4.070588,0.305882,2.476471,0.0000,0.0000,0,4,0,...,6.00,6.00,6.00,6.00,6.00,6.00,111.00,109.00,109.00,109.00
1,2016-01-01 01:00:00,108.00,4.305556,0.305556,2.211111,0.0000,0.0000,1,4,0,...,6.00,6.00,6.00,6.00,6.00,111.00,109.00,109.00,109.00,6.00
2,2016-01-01 02:00:00,108.00,3.655556,0.038889,1.533333,0.0000,0.0000,2,4,0,...,6.00,6.00,6.00,6.00,111.00,109.00,109.00,109.00,6.00,7.00
3,2016-01-01 03:00:00,108.00,3.477778,-0.294444,2.183333,0.0000,0.0000,3,4,0,...,6.00,6.00,6.00,111.00,109.00,109.00,109.00,6.00,7.00,7.00
4,2016-01-01 04:00:00,109.00,3.488889,-0.527778,2.383333,0.0000,0.0000,4,4,0,...,6.00,6.00,111.00,109.00,109.00,109.00,6.00,7.00,7.00,7.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1232605,2017-08-29 18:00:00,134.81,23.270588,14.888235,3.770588,13294.6148,858.9032,18,1,0,...,1036.71,609.52,141.81,139.70,141.22,142.12,139.98,135.55,131.53,181.11
1232606,2017-08-29 19:00:00,190.56,21.694118,14.664706,3.870588,11834.9156,858.9032,19,1,0,...,609.52,141.81,139.70,141.22,142.12,139.98,135.55,131.53,181.11,194.93
1232607,2017-08-29 20:00:00,199.67,20.331579,14.268421,2.573684,10171.2644,858.9032,20,1,0,...,141.81,139.70,141.22,142.12,139.98,135.55,131.53,181.11,194.93,243.45
1232608,2017-08-29 21:00:00,285.35,19.784211,14.068421,3.173684,7722.3541,0.0000,21,1,0,...,139.70,141.22,142.12,139.98,135.55,131.53,181.11,194.93,243.45,1654.64


In [8]:
import time
# import xgboost as xgb
import lightgbm as lgb

forecast_horizon = 24

for h in range(forecast_horizon):
    times = time.time()
    model = lgb.LGBMRegressor(
        device="gpu",
        n_estimators=100
    )# defaut 100
    model.fit(data_frame[X_cols], data_frame[f"target_t+{h+1}"])

    joblib.dump(model, f"{model_dir}/model_hour_{h+1}.pkl")
    print(f"Training time model {h} {time.time() - times}:", )

Exception in thread Thread-4 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\tuong\anaconda3\envs\test\Lib\threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "c:\Users\tuong\anaconda3\envs\test\Lib\threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\tuong\anaconda3\envs\test\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "<frozen codecs>", line 322, in decode
UnicodeDecodeError: 'utf-8' codec can't decode byte 0x88 in position 40: invalid start byte


[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 1507
[LightGBM] [Info] Number of data points in the train set: 1232610, number of used features: 13
[LightGBM] [Info] Using GPU Device: Intel(R) UHD Graphics, Vendor: Intel(R) Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 12 dense feature groups (14.11 MB) transferred to GPU in 0.018146 secs. 1 sparse feature groups
[LightGBM] [Info] Start training from score 174.348168
Training time model 0 11.653945684432983:
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 1507
[LightGBM] [Info] Number of data points in the train set: 1232610, number of used features: 13
[LightGBM] [Info] Using GPU Device: Intel(R) UHD Graphics, Vendor: Intel(R) Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[Light

In [ ]:
# import time
# import xgboost as xgb

# forecast_horizon = 24
# times = time.time()
# for h in range(forecast_horizon):
#     model = xgb.XGBRegressor(
#         tree_method="gpu_hist",   
#         predictor="gpu_predictor",
#         gpu_id=0,
#         n_estimators=300,
#         max_depth=6,
#         learning_rate=0.05,
#         subsample=0.8,
#         colsample_bytree=0.8,
#         objective="reg:squarederror",
#         random_state=42
#     )
    
#     target_col = f"target_t+{h+1}"
#     df_train = data_frame[X_cols + [target_col]].dropna()
#     print(f"Training model for horizon {h+1}, training samples: {len(df_train)}")
#     X_train = df_train[X_cols]
#     y_train = df_train[target_col]
    
#     model.fit(X_train, y_train)
#     joblib.dump(model, f"{model_dir}/model_hour_{h+1}.pkl")
# print("Training time:", time.time() - times)